# TextRank Summarizer - Đánh giá

Notebook này dùng để chạy thử mô hình TextRank và đánh giá điểm ROUGE trên tập dữ liệu tiếng Việt.
Việc sử dụng Pandas DataFrame giúp hiển thị kết quả trực quan và dễ so sánh hơn.


In [5]:
import sys
import os
import json
import pandas as pd
from IPython.display import display, HTML
from rouge_score import rouge_scorer
from textrank import TextRankSummarizer

# Cấu hình Pandas để hiển thị text dài không bị cắt bớt
pd.set_option('display.max_colwidth', None)

## 1. Tải dữ liệu
Đọc 10 mẫu dữ liệu từ file `summarization_samples.json`.

In [6]:
DATA_PATH = "../data/summarization_samples.json"

with open(DATA_PATH, "r", encoding="utf-8") as f:
    samples = json.load(f)

print(f"Loaded {len(samples)} samples.")

Loaded 10 samples.


## 2. Định nghĩa hàm đánh giá
Hàm tính toán ROUGE score (ROUGE-1, ROUGE-2, ROUGE-L).

In [7]:
def evaluate_rouge(prediction: str, reference: str) -> dict:
    scorer = rouge_scorer.RougeScorer(["rouge1", "rouge2", "rougeL"], use_stemmer=False)
    scores = scorer.score(reference, prediction)
    return {
        "ROUGE-1": round(scores["rouge1"].fmeasure, 4),
        "ROUGE-2": round(scores["rouge2"].fmeasure, 4),
        "ROUGE-L": round(scores["rougeL"].fmeasure, 4),
    }

## 3. Chạy tóm tắt và đánh giá
Chạy TextRank với `ratio=0.5` cho từng mẫu và lưu kết quả vào danh sách.

In [8]:
summarizer = TextRankSummarizer(ratio=0.5)

results = []
ratio_str = os.environ.get("SUMMARIZATION_RATIO", "0.5")
ratio = float(ratio_str)

print(ratio)

for sample in samples:
    text = sample["text"]
    ref_summary = sample["summary"]
    
    # Generate summary
    predicted_summary = summarizer.summarize(text,ratio)
    
    # Evaluate
    rouge = evaluate_rouge(predicted_summary, ref_summary)
    
    results.append({
        "ID": sample["id"],
        "Original Text": text,
        "Reference Summary": ref_summary,
        "TextRank Summary": predicted_summary,
        "ROUGE-1": rouge["ROUGE-1"],
        "ROUGE-2": rouge["ROUGE-2"],
        "ROUGE-L": rouge["ROUGE-L"]
    })

0.3


## 4. Xem kết quả
Hiển thị điểm trung bình và bảng chi tiết.

In [ ]:
df = pd.DataFrame(results)

# Hiển thị điểm trung bình
print("OVERALL AVERAGE SCORES:")
display(pd.DataFrame(df[["ROUGE-1", "ROUGE-2", "ROUGE-L"]].mean(), columns=["Average Score"]).T)

print("\n\n CHI TIẾT TỪNG MẪU:")
# Tạo bản sao để hiển thị cho gọn phần Original Text (cắt bớt nếu quá dài)
df_display = df.copy()
df_display["Original Text"] = df_display["Original Text"].apply(lambda x: x[:150] + "..." if len(x) > 150 else x)

# Sử dụng style để text hiển thị rõ ràng hơn
styled_df = df_display.style.set_properties(**{
    'text-align': 'left',
    'white-space': 'pre-wrap',
}).set_table_styles([
    dict(selector='th', props=[('text-align', 'center'), ('background-color', '#f4f4f4')])
])

display(styled_df)

📊 OVERALL AVERAGE SCORES:


,ROUGE-1,ROUGE-2,ROUGE-L
Average Score,0.61953,0.25762,0.38467




📝 CHI TIẾT TỪNG MẪU:


,ID,Original Text,Reference Summary,TextRank Summary,ROUGE-1,ROUGE-2,ROUGE-L
0,1,Nghiên cứu mới đây từ Đại học Oxford cho thấy việc ngủ đủ 7-8 tiếng mỗi đêm không chỉ giúp phục hồi thể lực mà còn đóng vai trò quan trọng trong việc ...,"Ngủ đủ 7-8 tiếng mỗi đêm giúp não bộ dọn dẹp chất thải chuyển hóa, củng cố trí nhớ và giảm nguy cơ suy giảm trí nhớ, mất tập trung.",Những người thường xuyên thiếu ngủ có nguy cơ cao mắc các bệnh suy giảm trí nhớ và khó tập trung hơn 30% so với người ngủ đủ giấc.,0.597700,0.211800,0.390800
1,2,Trí tuệ nhân tạo (AI) đang tạo ra một cuộc cách mạng trong lĩnh vực y tế. Bằng cách phân tích hàng triệu bệnh án và hình ảnh y khoa trong thời gian ng...,"AI đang cách mạng hóa ngành y tế nhờ khả năng phân tích dữ liệu nhanh chóng và chẩn đoán chính xác ung thư, hứa hẹn trở thành công cụ hỗ trợ đắc lực cho bác sĩ.","Tuy nhiên, các chuyên gia nhấn mạnh AI sẽ không thay thế bác sĩ mà đóng vai trò như một công cụ hỗ trợ đắc lực.",0.587000,0.222200,0.347800
2,3,Biến đổi khí hậu đang gây ra những tác động nghiêm trọng đến hệ sinh thái toàn cầu. Nhiệt độ Trái Đất tăng cao làm băng ở hai cực tan chảy với tốc độ ...,"Biến đổi khí hậu làm tăng nhiệt độ, tan băng, nước biển dâng và gây ra nhiều thời tiết cực đoan, đe dọa nghiêm trọng đến đời sống con người và hệ sinh thái.","Nhiệt độ Trái Đất tăng cao làm băng ở hai cực tan chảy với tốc độ kỷ lục, dẫn đến mực nước biển dâng đe dọa các thành phố ven biển.",0.620000,0.285700,0.320000
3,4,Chế độ ăn Địa Trung Hải từ lâu đã được công nhận là một trong những lối sống lành mạnh nhất thế giới. Chế độ ăn này tập trung vào việc tiêu thụ nhiều ...,"Chế độ ăn Địa Trung Hải ưu tiên rau củ, dầu oliu, hải sản và hạn chế thịt đỏ giúp tăng tuổi thọ, giảm nguy cơ mắc bệnh tim mạch và tiểu đường.","Các nghiên cứu kéo dài nhiều thập kỷ chứng minh rằng việc tuân thủ chế độ ăn này giúp giảm đáng kể nguy cơ mắc bệnh tim mạch, tiểu đường tuýp 2 và tăng cường tuổi thọ.",0.653800,0.333300,0.442300
4,5,Kính viễn vọng không gian James Webb của NASA đã gửi về những hình ảnh sắc nét chưa từng có về một cụm thiên hà cách Trái Đất hàng tỷ năm ánh sáng. Nh...,"Kính viễn vọng James Webb gửi về hình ảnh chi tiết của cụm thiên hà xa xôi, giúp tìm hiểu vũ trụ sơ khai và phát hiện dấu hiệu có thể có sự sống trên các ngoại hành tinh.",Kính viễn vọng không gian James Webb của NASA đã gửi về những hình ảnh sắc nét chưa từng có về một cụm thiên hà cách Trái Đất hàng tỷ năm ánh sáng.,0.672600,0.306300,0.477900
5,6,Xe điện đang dần chiếm lĩnh thị trường ô tô toàn cầu nhờ vào sự cải tiến vượt bậc của công nghệ pin và sự gia tăng các trạm sạc. So với xe động cơ đốt...,"Xe điện ngày càng phổ biến nhờ thân thiện với môi trường và chi phí bảo dưỡng thấp, nhưng vẫn phải đối mặt với thách thức về thời gian sạc và tái chế pin.",Xe điện đang dần chiếm lĩnh thị trường ô tô toàn cầu nhờ vào sự cải tiến vượt bậc của công nghệ pin và sự gia tăng các trạm sạc.,0.582500,0.158400,0.330100
6,7,"Tập thể dục nhịp điệu (aerobic) không chỉ tốt cho tim mạch mà còn mang lại những lợi ích to lớn cho sức khỏe tinh thần. Khi vận động, cơ thể giải phón...","Tập thể dục nhịp điệu kích thích tiết endorphin, giúp giảm căng thẳng, lo âu và là phương pháp hiệu quả để cải thiện tâm trạng cho người trầm cảm nhẹ.",Tập thể dục nhịp điệu (aerobic) không chỉ tốt cho tim mạch mà còn mang lại những lợi ích to lớn cho sức khỏe tinh thần.,0.565200,0.200000,0.413000
7,8,"Rạn san hô Great Barrier ở Úc, hệ thống đá ngầm san hô lớn nhất thế giới, đang phải đối mặt với hiện tượng tẩy trắng hàng loạt do nhiệt độ nước biển n...","Nhiệt độ nước biển tăng cao làm rạn san hô Great Barrier bị tẩy trắng hàng loạt do mất đi tảo cộng sinh, đe dọa nghiêm trọng đến hệ sinh thái và đa dạng sinh học biển.","Rạn san hô Great Barrier ở Úc, hệ thống đá ngầm san hô lớn nhất thế giới, đang phải đối mặt với hiện tượng tẩy trắng hàng loạt do nhiệt độ nước biển nóng lên bất thường.",0.732100,0.400000,0.375000
8,9,Học một ngoại ngữ mới mang lại nhiều lợi ích kognitive (nhận th